# Semana 36 · ¿Qué queda cuando muere una estrella?

Este cuaderno acompaña a la sesión de proyecto de la semana 36: el radio de Schwarzschild, las densidades de todo lo visto en la fase, la relación masa-radio de las enanas blancas, las mareas cerca de un agujero negro y el periodo mínimo de un púlsar.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El radio de Schwarzschild

$$r_s = \frac{2GM}{c^2}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G, c = 6.674e-11, 2.998e8
M_SOL, R_SOL = 1.989e30, 6.957e8
UA = 1.496e11

def radio_schwarzschild(M_kg):
    return 2 * G * M_kg / c**2

def legible(r_m):
    if r_m < 1e-3: return f"{r_m:.1e} m"
    if r_m < 1: return f"{r_m * 1000:.1f} mm"
    if r_m < 1e3: return f"{r_m:.1f} m"
    if r_m < 1e9: return f"{r_m / 1e3:,.1f} km".replace(",", " ")
    return f"{r_m / UA:.3g} ua"

objetos = [("una persona", 70), ("la Tierra", 5.97e24), ("Júpiter", 1.898e27), ("el Sol", M_SOL),
           ("agujero negro estelar (10 M☉)", 10 * M_SOL), ("Sagitario A*", 4.3e6 * M_SOL), ("M87*", 6.5e9 * M_SOL)]
for nombre, M in objetos:
    print(f"{nombre:<32} r_s = {legible(radio_schwarzschild(M))}")

## 2. Densidades, de la nube al agujero negro

Una escala logarítmica con todo lo que ha salido en la fase 3. Los valores son órdenes de magnitud (para el agujero negro, la densidad media dentro del horizonte, que depende de la masa).

In [ ]:
def densidad_media(M, R):
    return M / (4 / 3 * np.pi * R**3)

densidades = {
    "núcleo de nube molecular": 3.8e-17,
    "aire": 1.2,
    "agua": 1000,
    "Sol (media)": densidad_media(M_SOL, R_SOL),
    "centro del Sol": 1.5e5,
    "enana blanca (0,6 M☉)": densidad_media(0.6 * M_SOL, 8.7e6),
    "núcleo atómico": 2.3e17,
    "estrella de neutrones": densidad_media(1.4 * M_SOL, 1.2e4),
    "agujero negro de 10 M☉": densidad_media(10 * M_SOL, radio_schwarzschild(10 * M_SOL)),
    "Sagitario A*": densidad_media(4.3e6 * M_SOL, radio_schwarzschild(4.3e6 * M_SOL)),
    "M87*": densidad_media(6.5e9 * M_SOL, radio_schwarzschild(6.5e9 * M_SOL)),
}
nombres = list(densidades)
plt.figure(figsize=(9, 5.5))
plt.barh(nombres, list(densidades.values()), color="tab:purple")
plt.xscale("log"); plt.xlabel("densidad (kg/m³, escala logarítmica)"); plt.grid(alpha=0.3, axis="x")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

Unos 34 órdenes de magnitud entre un núcleo de nube molecular y una estrella de neutrones. Y fíjate en los agujeros negros: cuanto más masivos, menos densos (de media). M87* es menos denso que el aire.

## 3. Las enanas blancas: más masa, menos radio

Una enana blanca se sostiene por la presión de degeneración de sus electrones. Una consecuencia sorprendente: cuanta más masa tiene, **más pequeña** es, porque la gravedad la comprime más. En 1972, Michael Nauenberg publicó una fórmula aproximada que resume los cálculos de Chandrasekhar:

$$\frac{R}{R_\odot} \approx 0{,}0112\,\sqrt{\left(\frac{M}{M_\text{Ch}}\right)^{-2/3} - \left(\frac{M}{M_\text{Ch}}\right)^{2/3}}, \qquad M_\text{Ch} \approx 1{,}44\,M_\odot.$$

El radio se hace cero en el límite de Chandrasekhar: por encima no hay enana blanca posible.

In [ ]:
M_CH = 1.44
M = np.linspace(0.1, 1.44, 400)
R = 0.0112 * np.sqrt((M / M_CH) ** (-2 / 3) - (M / M_CH) ** (2 / 3))

plt.figure(figsize=(8, 5))
plt.plot(M, R * R_SOL / 1e3, label="fórmula de Nauenberg")
plt.axhline(6371, color="tab:green", ls="--", label="radio de la Tierra")
plt.plot(1.02, 0.0084 * R_SOL / 1e3, "ro", label="Sirio B (observada)")
plt.plot(0.6, 8.7e3, "ks", label="enana blanca típica")
plt.xlabel("masa (M☉)"); plt.ylabel("radio (km)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Relación masa-radio de las enanas blancas"); plt.show()

## 4. Mareas cerca de un agujero negro

La diferencia de aceleración entre la cabeza y los pies de una persona de 2 m que cae hacia un agujero negro, $\Delta a \approx 2GM\ell/r^3$, en función de la distancia al centro. Marcamos el horizonte de cada uno y la aceleración que, por ejemplo, empezaría a ser dañina (unas 10 veces la gravedad terrestre, como referencia aproximada).

In [ ]:
ell = 2.0
fig, ax = plt.subplots(figsize=(9, 5.5))
for nombre, M, color in [("10 M☉", 10 * M_SOL, "tab:red"), ("Sagitario A*", 4.3e6 * M_SOL, "tab:blue")]:
    rs = radio_schwarzschild(M)
    r = np.logspace(np.log10(rs), np.log10(rs) + 4, 300)
    ax.loglog(r / 1e3, 2 * G * M * ell / r**3, color=color, label=nombre)
    ax.axvline(rs / 1e3, color=color, ls=":", lw=1)
    print(f"{nombre}: marea en el horizonte = {2 * G * M * ell / rs**3:.2e} m/s²")
ax.axhline(98, color="gray", ls="--", label="unas 10 g")
ax.set_xlabel("distancia al centro (km)"); ax.set_ylabel("Δa entre cabeza y pies (m/s²)")
ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()

Junto al agujero negro pequeño, la marea es destructiva miles de kilómetros antes de llegar al horizonte. En Sagitario A*, en el horizonte es menor que la de una montaña rusa.

## 5. El periodo de giro mínimo

$$P_\text{mín} = \sqrt{\frac{3\pi}{G\bar\rho}}.$$

Un objeto que girara más deprisa se desharía. Lo calculamos para todo lo que gira.

In [ ]:
for nombre, rho in [("la Tierra", 5500), ("el Sol", densidad_media(M_SOL, R_SOL)),
                    ("una enana blanca", densidad_media(0.6 * M_SOL, 8.7e6)),
                    ("una estrella de neutrones", densidad_media(1.4 * M_SOL, 1.2e4))]:
    P = np.sqrt(3 * np.pi / (G * rho))
    texto = f"{P / 3600:.1f} horas" if P > 3600 else (f"{P:.1f} s" if P > 1 else f"{P * 1000:.2f} ms")
    print(f"{nombre:<28} P mínimo ≈ {texto}")
print("púlsar del Cangrejo: 33 ms; púlsar más rápido conocido: 1,4 ms")

## 6. Experimenta

1. ¿Qué masa tendría que tener un agujero negro para que su radio de Schwarzschild fuera de 1 ua? ¿Y para que la marea en su horizonte fuera de 9,8 m/s² con $\ell = 2$ m?
2. Con la fórmula de Nauenberg, ¿qué masa tiene una enana blanca del tamaño exacto de la Tierra?
3. Calcula la velocidad de escape de cada objeto de la sección 1 en su superficie real (busca sus radios) y compárala con $c$.

In [ ]:
# Tu experimento aquí